In [1]:
import argparse
import io
import os
import time
import warnings

import azure.storage.blob
import numpy as np
import pandas as pd
import planetary_computer
import pystac_client
import rioxarray  # rioxarray is required for the .rio methods in xarray despite what mypy, ruff, etc. says :)
import stackstac
from tqdm import tqdm

In [3]:
parquet_file = "/mnt/rolf-datastore/home/leca5365/Documents/satclip/notebooks/s2l2a_2_5_2024_clouds_lt_20_date_gt_2021_01_01.parquet"

df = pd.read_parquet(parquet_file)
num_rows = df.shape[0]

test_id = df.iloc[0]["id"]
test_id

'S2B_MSIL2A_20241201T002959_R059_T56RMN_20241201T013520'

In [4]:
# Connect to Microsoft Planetary Computer STAC API
catalog = pystac_client.Client.open(
    "https://planetarycomputer.microsoft.com/api/stac/v1/",
    modifier=planetary_computer.sign_inplace,
)

collection = catalog.get_collection("sentinel-2-l2a")

item = collection.get_item(df.iloc[0]["id"])

with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    stack = stackstac.stack(
        item,
        assets=[
            "B01",
            "B02",
            "B03",
            "B04",
            "B05",
            "B06",
            "B07",
            "B08",
            "B8A",
            "B09",
            "B11",
            "B12",
        ],
        epsg=4326,
    )
_, num_channels, height, width = stack.shape

# Randomly sample a 256x256 window within image bounds
x = np.random.randint(0, width - 256)
y = np.random.randint(0, height - 256)

# Extract patch and compute in-memory
# patch = stack[0, :, y : y + 256, x : x + 256].compute()
patch = stack[0, :, y : y + 256, x : x + 256].compute()

CRSError: The EPSG code is unknown. PROJ: internal_proj_create_from_database: /home/leca5365/miniconda3/envs/satclip313/share/proj/proj.db contains DATABASE.LAYOUT.VERSION.MINOR = 2 whereas a number >= 6 is expected. It comes from another PROJ installation.